In [1]:
import os

In [9]:
import pandas as pd
import glob
import os

# 1. Setup path and find all relevant CSV files
path = './storm_data/konalow20260316'  # Change this to the folder containing your CSVs
all_files = glob.glob(os.path.join(path, "RFdata_*.csv"))

# Initialize an empty list or a starting DataFrame
master_df = pd.DataFrame()

for file_path in all_files:
    # 2. Extract station name from filename (e.g., RFdata_..._Piiholo.csv -> Piiholo)
    filename = os.path.basename(file_path)
    station_name = filename.split('_')[-2].replace('.csv', '')
    
    # 3. Read the CSV
    # We only need 'timestamp' and 'RF_1_cum_in'
    df = pd.read_csv(file_path, usecols=['timestamp', 'RF_1_cum_in'])
    
    # Convert timestamp to datetime objects for cleaner merging
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Rename the data column to the station name
    df = df.rename(columns={'RF_1_cum_in': station_name})
    
    # 4. Merge into the master DataFrame
    if master_df.empty:
        master_df = df
    else:
        master_df = pd.merge(master_df, df, on='timestamp', how='outer')

# 5. Sort by time and save
master_df = master_df.sort_values(by='timestamp').reset_index(drop=True)
# master_df.to_csv('merged_rf_data.csv', index=False)

print("Merge complete! Here's a preview:")
print(master_df.head())
master_df.to_csv('merged_storm_data.csv', index=False)

Merge complete! Here's a preview:
            timestamp  0257  0501  0153  0116  0255  0601  0121  0431  0541  \
0 2026-03-10 00:00:00   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
1 2026-03-10 00:05:00   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
2 2026-03-10 00:10:00   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
3 2026-03-10 00:15:00   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
4 2026-03-10 00:20:00   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   

   ...  0641  0242  0551  0503  0531  0256  0506  0254  0204  0144  
0  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
1  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
2  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
3  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
4  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  

[5 rows x 74 columns]


In [4]:
master_df.columns

Index(['timestamp', '0257', '0501', '0153', '0116', '0255', '0601', '0121',
       '0431', '0541', '0412', '0611', '0156', '0131', '0145', '0421', '0504',
       '0288', '0602', '0283', '0258', '0521', '0214', '0603', '0231', '0211',
       '0621', '0253', '0243', '0235', '0132', '0213', '0282', '0251', '0202',
       '0411', '0502', '0281', '0161', '0155', '0160', '0201', '0252', '0134',
       '0118', '0133', '0286', '0141', '0222', '0152', '0115', '0505', '0221',
       '0165', '0154', '0151', '0246', '0119', '0166', '0122', '0241', '0287',
       '0212', '0422', '0641', '0242', '0551', '0503', '0531', '0256', '0506',
       '0254', '0204', '0144'],
      dtype='object')

In [6]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# 1. Get the list of stations (all columns except 'timestamp')
stations = [col for col in master_df.columns if col != 'timestamp']
num_stations = len(stations)

# 2. Generate "rainbow" colors
# We use the 'gist_rainbow' or 'hsv' colormap to get a full spectrum
colormap = plt.get_cmap('gist_rainbow')
colors = [colormap(i) for i in np.linspace(0, 1, num_stations)]

# 3. Convert RGBA colors to Hex codes and map them to station IDs
station_colors = {}
for i, station in enumerate(stations):
    hex_code = mcolors.to_hex(colors[i])
    station_colors[station] = hex_code

# Output the results
print(f"Total number of stations: {num_stations}")
print("Station Color Mapping:")
color_df = pd.DataFrame.from_dict(station_colors, orient='index', columns=['Hex_Color'])
color_df.index.name = 'Station_ID'
color_df.reset_index(inplace=True)

Total number of stations: 73
Station Color Mapping:


In [8]:
color_df.to_csv('station_color_mapping.csv', index=False)